<sub>This is an example notebook on how to use the ncwm library via .ipynb notebooks (without the scripts in src/)</sub>

First of all, to train a model a dataset is needed, for this example I included a tiny dataset in the folder `example/data/`, all the files are .npz files containing the collected states and actions of the environment.

[Click here to proceed with training](#training)  
Alternatively, proceed here to understand how the dataset is generated and the states and actions' shapes.

The environment was made following a gym-like interface.  
The cell below prints the shape of the state of the environment before any action:

In [ ]:
from env.env import ExampleEnv

env = ExampleEnv()
state = env.reset() # reset returns the initial state

print(f"State shape: (H, W) - {state.shape}")

The environment is extremely simple and solveable without any AI, but for the purpose of this example, here's a short description of it:

- The environment is a 8x8 grid  
- Two classes: Background (ID 0) and Player (ID 1)  
- Four actions: Up, Down, Left, Right, in this order, from 0 to 3  
- Player's position is randomized each reset(), X and Y both within the range [1, 7]  
- If the action causes the player to be outside the grid, the player won't move  
- The environment is deterministic (excluding the random player spawn position) and Markovian  

Here is an example of how you can generate a dataset of this environment:

In [ ]:
from env.env import ExampleEnv
import numpy as np

from tqdm import tqdm

env = ExampleEnv()

EPISODES = 10
STEPS = 400
OUT = "data/example_{:03d}.npz"

# define how an episode is generated
def make_episode(steps: int):
    states = np.empty((steps, 8, 8), dtype=np.uint8) # BHW
    actions = np.empty((steps-1), dtype=np.uint8) # B

    state = env.reset()

    for i in range(steps-1):
        action = np.random.randint(0, 4) # [0, 4)

        states[i] = state
        actions[i] = action

        state = env.step(action)
    
    states[-1] = state

    return states, actions

# creating dataset (multiprocessing can be used for slower environments)
for i in tqdm(range(EPISODES), "Making episodes"):
    states, actions = make_episode(STEPS)

    # save
    np.savez_compressed(OUT.format(i), states=states, actions=actions)

print(f"Done! - Generated {EPISODES * STEPS} states")

If done correctly, the shape of 'states' should be (B, H, W) where B is STEPS, H and W are height and width of the environment (8x8 by default) and the values will range from 0 to 1 (class background or player), here's an example of what the first state looks like:

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt

BG_COLOR = "#212121" # dark gray
PLAYER_COLOR = "#F0F0F0" # white

cmap = mcolors.ListedColormap([PLAYER_COLOR, BG_COLOR])
plt.figure(facecolor=BG_COLOR) 

plt.imshow(states[0], cmap=cmap, interpolation="nearest")
plt.axis("off");

## Training

To train, inference or benchmark a model, the library `ncwm` can be used (which contents are in `../ncwm/`)  
Before anything, a configuration file must be created, (as example, [`config.py`](config.py)).

Below is a guided walkthrough in creating a configuration:

In [ ]:
# training-related
STEPS = 500
BATCH_SIZE = 64
LOG_SEGMENTS = 100 # how many times to log during training (100 means every 5 steps in this case)
LOAD_MODEL = None # path to a previously saved model to continue training
SAVE_MODEL = "model.eqx" # where to save the model
LOSS_GRAPH = "loss_graph.png" # where to save the loss graph
TRUNCATED_BPTT = 1 # not necessary for this example (as the environment is Markovian)

In [ ]:
# data-related
DATA_GLOB = "data/example_*.npz" # glob pattern of data files used for training
DATA_LIMIT = None # None means all files will be included, look at ncwm/base_config.py for more info
LOADING_MODE = "VRAM" # "VRAM" or "RAM" or "DISK", if device is CPU, "VRAM" will be interpreted as "RAM"

In [ ]:
# for inference
LOAD_MODEL_INF = "model.eqx" # model to load
LOAD_DATA_INF = "data/example_000.npz" # which data file to load as the first state, can be .npz, .npy or .png
DATA_IDX_INF = 0 # which index of the data file to load as the first state (if applicable)
WIN_SIZE = None # if None it will be calculated automatically based on the loaded data shape (height and width)
KEY_MAP = { # key to action mapping, actions are what the model expects as external input (converted to one-hot during model step)
    'w': 0,
    's': 1,
    'a': 2,
    'd': 3
}
DEFAULT_ACTION = None # default action (useful when FPS is set)
FPS = None # if None, the simulation will wait for inputs after every step

In [ ]:
# model-related
SUBSTEPS = 2 # how many forward passes per step
# higher values of SUBSTEPS allows the information to propagate further away between pixels
# more SUBSTEPS is quite similar (as in learning capacity) to adding more layers to the model

# COLOR_MAP to represent states as images (order matters)
COLOR_MAP = [
    [33, 33, 33], # background (dark gray)
    [240, 240, 240] # player (white)
]

import jax, jax.numpy as jnp
from ncwm import NCWM

# as the background is 63 pixels and player is 1 pixel, player should be weighted 63 times
# this avoids the loss to drop instantly even though the model hasn't learned how the player moves
_class_weights = jnp.array([1.0, 63.0]) # (order matters)

# creating the actual model:
def make_model(key: jax.Array) -> NCWM:
    return NCWM(
        # neural network
        actions=len(KEY_MAP),
        vis_channels=len(COLOR_MAP),
        hid_channels=0, # as the environment is Markovian, the model can learn without additional info or hidden states required
        hidden_neurons=24, # number of hidden neurons in the first layer of the CNN
        padding_mode='zeros', # what cells see at the edge of the grid (with zeros they have a clear information that they are at the edge)
        dtype=jnp.bfloat16, # data type (float32 by default)
        key=key, # random key (leave as is)
    )

from typing import Optional
import optax

# optimizer (for training):
def make_optimizer() -> tuple[optax.GradientTransformation, Optional[optax.Schedule]]:
    return optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(learning_rate=0.05, weight_decay=1e-4)
    ), None

# loss function (for training):
def loss_calc(vis_preds: jnp.ndarray, hid_preds: None, targets: jnp.ndarray, actions: jnp.ndarray, infos: jnp.ndarray) -> jnp.ndarray:
    # BCHW to BHWC
    vis_preds = jnp.moveaxis(vis_preds, 1, -1) # visible predictions
    vis_targets = jnp.moveaxis(targets, 1, -1) # visible targets
    
    celoss = optax.softmax_cross_entropy(vis_preds, vis_targets)

    class_idx = jnp.argmax(vis_targets, axis=-1) # argmax of C
    weights = _class_weights[class_idx]

    # making the loss weighted
    celoss = jnp.sum(celoss * weights) / jnp.sum(weights)

    return celoss

As the configuration's variables were defined in this Kernel, instead of running `src/train.py` with a configuration path as argument, the configuration can be built directly by passing `globals()` to `load_configuration()`:

In [ ]:
from ncwm import train_model, inference_cv2, load_configuration

config = load_configuration(globals())

To launch the training:

In [ ]:
train_model(config);

Finally, the model can be tested interactively with `inference_cv2.py`, exported as function 'inference_cv2' in ncwm:

In [ ]:
inference_cv2(config);